# Data preprocessing for comfort labels
This notebook loads and cleans these indoor comfort files:
- March28Data_W_IRRA(in).csv
- April4_9_10(April4_9_10).csv

It strips unit symbols, parses timestamps, standardizes labels, and exports one merged clean dataset.

In [1]:
from pathlib import Path

import pandas as pd

base = Path.cwd()

files = [
    base / "March28Data_W_IRRA(in).csv",
    base / "April4_9_10(April4_9_10).csv",
]

cols = [
    "Date",
    "Time",
    "IndoorTemp_raw",
    "IndoorRH_raw",
    "WallTemp_raw",
    "Ambient2_raw",
    "Label_raw",
]

def extract_number(series: pd.Series) -> pd.Series:
    return pd.to_numeric(
        series.astype(str).str.extract(r"(-?\d+(?:\.\d+)?)", expand=False),
        errors="coerce",
    )

frames = []
for fp in files:
    df = pd.read_csv(fp, header=None, names=cols)
    df["source_file"] = fp.name

    df["IndoorTemp_F"] = extract_number(df["IndoorTemp_raw"] )
    df["IndoorRH_percent"] = extract_number(df["IndoorRH_raw"] )
    df["WallTemp_F"] = extract_number(df["WallTemp_raw"] )
    df["Ambient2_F"] = extract_number(df["Ambient2_raw"] )
    df["IndoorRH_frac"] = df["IndoorRH_percent"] / 100.0

    df["Label"] = (
        df["Label_raw"]
        .astype(str)
        .str.strip()
        .str.title()
        .replace({"Comfort": "Good"})
    )

    dt_text = df["Date"].astype(str).str.strip() + "-2026 " + df["Time"].astype(str).str.strip()
    df["timestamp"] = pd.to_datetime(dt_text, format="%d-%b-%Y %H:%M:%S", errors="coerce")

    frames.append(df)

comfort_df = pd.concat(frames, ignore_index=True)

comfort_clean = comfort_df[[
    "timestamp",
    "Label",
    "IndoorTemp_F",
    "IndoorRH_percent",
    "IndoorRH_frac",
    "WallTemp_F",
    "Ambient2_F",
    "source_file",
]].copy()

comfort_clean = comfort_clean.dropna(subset=["timestamp", "Label", "IndoorTemp_F", "IndoorRH_percent"] )
comfort_clean = comfort_clean.sort_values("timestamp").drop_duplicates().reset_index(drop=True)

out_path = base / "comfort_labels_clean.csv"
comfort_clean.to_csv(out_path, index=False)

print(f"Rows after cleaning: {len(comfort_clean):,}")
# print(f"Saved: {out_path.name}")
print("\nLabel distribution:")
print(comfort_clean["Label"].value_counts(dropna=False))

comfort_clean.head()

Rows after cleaning: 8,304

Label distribution:
Label
Good    6017
Cold    1791
Hot      496
Name: count, dtype: int64


,timestamp,Label,IndoorTemp_F,IndoorRH_percent,IndoorRH_frac,WallTemp_F,Ambient2_F,source_file
0,2026-03-28 23:50:08,Good,68.54,26.9,0.269,72,72,March28Data_W_IRRA(in).csv
1,2026-03-28 23:50:10,Good,69.26,26.2,0.262,72,72,March28Data_W_IRRA(in).csv
2,2026-03-28 23:50:12,Good,69.26,26.2,0.262,73,72,March28Data_W_IRRA(in).csv
3,2026-03-28 23:50:14,Good,69.26,26.2,0.262,73,72,March28Data_W_IRRA(in).csv
4,2026-03-28 23:50:16,Good,69.26,26.2,0.262,73,72,March28Data_W_IRRA(in).csv


In [2]:
print("Label counts by source file:")
display(pd.crosstab(comfort_clean["source_file"], comfort_clean["Label"]))

print("\nNumeric summary:")
display(comfort_clean[["IndoorTemp_F", "IndoorRH_percent", "WallTemp_F", "Ambient2_F"]].describe())

label_counts = comfort_clean["Label"].value_counts()
if len(label_counts) > 1 and label_counts.max() / label_counts.min() > 5:
    print("Class imbalance detected. Consider class weights or resampling before training.")
else:
    print("Class balance looks reasonable for a first model")

Label counts by source file:


Label,Cold,Good,Hot
source_file,,,
April4_9_10(April4_9_10).csv,80,5201,496
March28Data_W_IRRA(in).csv,1711,816,0



Numeric summary:


,IndoorTemp_F,IndoorRH_percent,WallTemp_F,Ambient2_F
count,8304.000000,8304.000000,8304.000000,8304.000000
mean,66.920650,35.904420,65.637042,66.869461
std,2.777194,4.832037,3.137079,3.200636
min,64.580000,26.100000,63.000000,64.000000
25%,64.760000,32.200000,63.000000,64.000000
50%,65.480000,33.700000,64.000000,65.000000
75%,68.360000,41.600000,68.000000,69.000000
max,75.380000,46.200000,90.000000,75.000000


Class imbalance detected. Consider class weights or resampling before training.


## Timeline alignment with 2022 weather logs
This section checks whether comfort-label timestamps line up with the weather logs collected in March/April 2022.

Method:
- Load `Weather and Temperature Logs(March 2022).csv` and `Weather and Temperature Logs(April 2022).csv`
- Parse weather timestamps
- Shift comfort timestamps to year 2022 for calendar/time-of-day alignment checks
- Compute nearest weather timestamp gap and overlap statistics

In [3]:
# Load 2022 weather logs
weather_march = pd.read_csv(base / "Weather and Temperature Logs(March 2022).csv")
weather_april = pd.read_csv(base / "Weather and Temperature Logs(April 2022).csv")

weather_2022 = pd.concat([weather_march, weather_april], ignore_index=True)
weather_2022["weather_timestamp"] = pd.to_datetime(weather_2022["Time Stamp"], errors="coerce")
weather_2022 = weather_2022.dropna(subset=["weather_timestamp"]).sort_values("weather_timestamp")

# Shift comfort labels to 2022 for timeline comparison (same month/day/time, different year)
comfort_2022 = comfort_clean.copy()
comfort_2022["comfort_timestamp_2022"] = comfort_2022["timestamp"].apply(
    lambda ts: ts.replace(year=2022) if pd.notna(ts) else pd.NaT
)
comfort_2022 = comfort_2022.dropna(subset=["comfort_timestamp_2022"]).sort_values("comfort_timestamp_2022")

# Find nearest weather timestamp for each comfort record
aligned = pd.merge_asof(
    comfort_2022[["comfort_timestamp_2022", "Label", "source_file"]],
    weather_2022[["weather_timestamp", "Current Weather", "Temp (C)", "Humidity"]],
    left_on="comfort_timestamp_2022",
    right_on="weather_timestamp",
    direction="nearest",
    tolerance=pd.Timedelta("45min")
)

aligned["abs_gap_minutes"] = (
    (aligned["comfort_timestamp_2022"] - aligned["weather_timestamp"]).abs().dt.total_seconds() / 60.0
)

matched = aligned["weather_timestamp"].notna()
match_rate = matched.mean() * 100
median_gap = aligned.loc[matched, "abs_gap_minutes"].median() if matched.any() else float("nan")
p90_gap = aligned.loc[matched, "abs_gap_minutes"].quantile(0.9) if matched.any() else float("nan")

print(f"Comfort rows checked: {len(aligned):,}")
print(f"Rows matched to weather within 45 min: {matched.sum():,} ({match_rate:.1f}%)")
print(f"Median nearest gap (minutes): {median_gap:.2f}")
print(f"90th percentile gap (minutes): {p90_gap:.2f}")

print("\nMatch rate by comfort source file:")
display(
    aligned.assign(matched=aligned["weather_timestamp"].notna())
    .groupby("source_file", as_index=False)["matched"]
    .mean()
    .assign(matched_pct=lambda d: (d["matched"] * 100).round(1))
    [["source_file", "matched_pct"]]
    .sort_values("matched_pct", ascending=False)
)

print("\nGap summary by label (matched rows only):")
gap_by_label = aligned.loc[matched].groupby("Label")["abs_gap_minutes"]
gap_summary = pd.DataFrame({
    "count": gap_by_label.size(),
    "mean": gap_by_label.mean(),
    "median": gap_by_label.median(),
    "p90": gap_by_label.quantile(0.9),
    "max": gap_by_label.max(),
})
display(gap_summary.sort_index())

print("\nSample aligned rows:")
display(
    aligned[[
        "comfort_timestamp_2022",
        "weather_timestamp",
        "abs_gap_minutes",
        "Label",
        "Current Weather",
        "Temp (C)",
        "Humidity",
    ]].head(10)
)

Comfort rows checked: 8,304
Rows matched to weather within 45 min: 8,304 (100.0%)
Median nearest gap (minutes): 5.53
90th percentile gap (minutes): 9.68

Match rate by comfort source file:


,source_file,matched_pct
0,April4_9_10(April4_9_10).csv,100.0
1,March28Data_W_IRRA(in).csv,100.0



Gap summary by label (matched rows only):


,count,mean,median,p90,max
Label,,,,,
Cold,1791,5.033296,5.083333,8.983333,10.000000
Good,6017,6.033225,5.633333,9.973333,19.983333
Hot,496,5.690759,5.800000,9.150000,10.000000



Sample aligned rows:


,comfort_timestamp_2022,weather_timestamp,abs_gap_minutes,Label,Current Weather,Temp (C),Humidity
0,2022-03-28 23:50:08,2022-03-28 23:55:00,4.866667,Good,Clear,0.5,36.792179
1,2022-03-28 23:50:10,2022-03-28 23:55:00,4.833333,Good,Clear,0.5,36.792179
2,2022-03-28 23:50:12,2022-03-28 23:55:00,4.800000,Good,Clear,0.5,36.792179
3,2022-03-28 23:50:14,2022-03-28 23:55:00,4.766667,Good,Clear,0.5,36.792179
4,2022-03-28 23:50:16,2022-03-28 23:55:00,4.733333,Good,Clear,0.5,36.792179
5,2022-03-28 23:50:18,2022-03-28 23:55:00,4.700000,Good,Clear,0.5,36.792179
6,2022-03-28 23:50:20,2022-03-28 23:55:00,4.666667,Good,Clear,0.5,36.792179
7,2022-03-28 23:50:22,2022-03-28 23:55:00,4.633333,Good,Clear,0.5,36.792179
8,2022-03-28 23:50:24,2022-03-28 23:55:00,4.600000,Good,Clear,0.5,36.792179
9,2022-03-28 23:50:26,2022-03-28 23:55:00,4.566667,Good,Clear,0.5,36.792179


## Physics-only vs recorded indoor signals
This section evaluates how well the physics model alone reproduces measured indoor humidity and wall temperature.

Setup used here:
- Setpoint input to the physics model = recorded indoor temperature (`IndoorTemp_F`)
- Outdoor inputs come from nearest matched 2022 weather row
- Building constants fixed at `R_wall=2.3`, `A_envelope=70`, `ACH=0.5`, `volume=30`

In [4]:
import contextlib
import importlib
import io

import numpy as np

# Suppress print statements
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    ss = importlib.import_module("our_steady_state_model")

# Build a clean comparison table from already time-aligned rows
comparison = aligned.copy()
comparison = comparison.dropna(subset=["Temp (C)", "Humidity", "weather_timestamp"]).copy()

# Join back measured indoor signals at matching shifted timestamp
measured_map = comfort_2022[[
    "comfort_timestamp_2022",
    "IndoorTemp_F",
    "IndoorRH_percent",
    "IndoorRH_frac",
    "WallTemp_F",
    "Ambient2_F",
    "Label",
    "source_file",
]].drop_duplicates()

comparison = comparison.merge(
    measured_map,
    on=["comfort_timestamp_2022", "Label", "source_file"],
    how="left"
)

comparison = comparison.dropna(subset=["IndoorTemp_F", "IndoorRH_frac", "WallTemp_F"]).copy()

# Outdoor weather inputs expected by physics model
comparison["T_external_F"] = ss.c_to_f(comparison["Temp (C)"] )
comparison["RH_external_frac"] = comparison["Humidity"] / 100.0

# building constants used in this analysis
R_wall = 2.3
A_envelope = 70
ACH = 0.5
volume = 30

# Run steady-state model row-by-row
pred_rh = []
pred_wall = []
pred_heat_loss = []
pred_user_q = []
pred_verdict = []

for _, row in comparison.iterrows():
    user_q_total, verdict, rh_internal, q_total, t_wall_f = ss.steady_state_model(
        T_setpoint_F=float(row["IndoorTemp_F"]),
        T_external_F=float(row["T_external_F"]),
        RH_external=float(row["RH_external_frac"]),
        R_wall=R_wall,
        A_envelope=A_envelope,
        ACH=ACH,
        volume=volume,
        print_output=False,
    )
    pred_user_q.append(user_q_total)
    pred_verdict.append(verdict)
    pred_rh.append(rh_internal)
    pred_heat_loss.append(q_total)
    pred_wall.append(t_wall_f)

comparison["RH_pred_frac"] = pred_rh
comparison["WallTemp_pred_F"] = pred_wall
comparison["BuildingHeatLoss_W"] = pred_heat_loss
comparison["UserHeatLoss_Wm2"] = pred_user_q
comparison["PhysicsVerdict"] = pred_verdict

# Error metrics
def metric_summary(actual, pred):
    err = pred - actual
    mae = np.mean(np.abs(err))
    rmse = np.sqrt(np.mean(err ** 2))
    bias = np.mean(err)
    corr = np.corrcoef(actual, pred)[0, 1] if len(actual) > 1 else np.nan
    return pd.Series({"MAE": mae, "RMSE": rmse, "Bias": bias, "Corr": corr})

rh_metrics = metric_summary(comparison["IndoorRH_frac"].to_numpy(), comparison["RH_pred_frac"].to_numpy())
wall_metrics = metric_summary(comparison["WallTemp_F"].to_numpy(), comparison["WallTemp_pred_F"].to_numpy())

print(f"Rows used for physics-vs-recorded comparison: {len(comparison):,}")
print("\nHumidity alignment metrics (fraction units):")
display(rh_metrics.to_frame(name="RH").T)

print("Wall temperature alignment metrics (F):")
display(wall_metrics.to_frame(name="WallTemp_F").T)

print("Per-label wall temperature MAE (F):")
display(
    comparison.assign(wall_abs_err=(comparison["WallTemp_pred_F"] - comparison["WallTemp_F"]).abs())
    .groupby("Label", as_index=False)["wall_abs_err"].mean()
    .rename(columns={"wall_abs_err": "WallTemp_MAE_F"})
    .sort_values("WallTemp_MAE_F")
)

print("Per-label humidity MAE (fraction):")
display(
    comparison.assign(rh_abs_err=(comparison["RH_pred_frac"] - comparison["IndoorRH_frac"]).abs())
    .groupby("Label", as_index=False)["rh_abs_err"].mean()
    .rename(columns={"rh_abs_err": "RH_MAE_frac"})
    .sort_values("RH_MAE_frac")
)

# Save for later if needed
comparison_out = base / "comfort_weather_physics_comparison.csv"
comparison_out = base / "comfort_weather_physics_comparison.csv"
comparison.to_csv(comparison_out, index=False)
# print(f"\nSaved comparison dataset: {comparison_out.name}")

display(
    comparison[[
        "comfort_timestamp_2022", "Label",
        "IndoorRH_frac", "RH_pred_frac",
        "WallTemp_F", "WallTemp_pred_F",
        "BuildingHeatLoss_W", "PhysicsVerdict",
    ]].head(10)
)

Rows used for physics-vs-recorded comparison: 6,201

Humidity alignment metrics (fraction units):


,MAE,RMSE,Bias,Corr
RH,0.120346,0.139868,-0.120346,-0.682548


Wall temperature alignment metrics (F):


,MAE,RMSE,Bias,Corr
WallTemp_F,0.520134,0.854491,-0.32823,0.973473


Per-label wall temperature MAE (F):


,Label,WallTemp_MAE_F
2,Hot,0.285991
1,Good,0.525693
0,Cold,0.602023


Per-label humidity MAE (fraction):


,Label,RH_MAE_frac
1,Good,0.082900
2,Hot,0.225845
0,Cold,0.228742


,comfort_timestamp_2022,Label,IndoorRH_frac,RH_pred_frac,WallTemp_F,WallTemp_pred_F,BuildingHeatLoss_W,PhysicsVerdict
0,2022-03-28 23:50:08,Good,0.269,0.097851,72,66.192968,702.103696,Good
1,2022-03-28 23:50:10,Good,0.262,0.095465,72,66.865553,716.287609,Good
2,2022-03-28 23:50:12,Good,0.262,0.095465,73,66.865553,716.287609,Good
3,2022-03-28 23:50:14,Good,0.262,0.095465,73,66.865553,716.287609,Good
4,2022-03-28 23:50:16,Good,0.262,0.095465,73,66.865553,716.287609,Good
5,2022-03-28 23:50:18,Good,0.262,0.095465,73,66.865553,716.287609,Good
6,2022-03-28 23:50:20,Good,0.262,0.095465,73,66.865553,716.287609,Good
7,2022-03-28 23:50:22,Good,0.262,0.095465,90,66.865553,716.287609,Good
8,2022-03-28 23:50:24,Good,0.261,0.095465,90,66.865553,716.287609,Good
9,2022-03-28 23:50:26,Good,0.261,0.095465,73,66.865553,716.287609,Good


## Interpretation of current fit
- High correlation does not mean close point-by-point agreement. The wall temperature series can still have large local residuals even when overall trend correlation is high.
- In this dataset, wall temperature has good global trend fit but noticeable local mismatches, including obvious spikes in measured wall values that the steady-state model does not reproduce.
- For indoor humidity, the physics output is not only lower on average (negative bias), it also shows weak to opposite trend tracking in this setup (negative correlation).
- Conclusion: wall prediction is useful as an engineered feature but not a direct substitute for sensor wall values; humidity prediction needs recalibration or model refinement before being treated as accurate.

In [5]:
# Build the final combined dataset for Random Forest training using source-based prefixes.
# This cell assumes previous preprocessing/alignment/physics cells have been run and `comparison` exists.


# 1) Start from the physics comparison table, which already contains:
#    - matched weather fields
#    - measured indoor fields
#    - steady-state model outputs
combined = comparison.copy()

# 2) Add/refresh building constants as explicit columns so each row is self-contained.
combined["R_wall"] = float(R_wall)
combined["A_envelope"] = float(A_envelope)
combined["ACH"] = float(ACH)
combined["volume"] = float(volume)

# 3) Ensure optional measured columns exist.
#    Some notebook runs may not carry these columns through `comparison`, so we backfill safely.
if "IndoorRH_percent" not in combined.columns and "IndoorRH_frac" in combined.columns:
    combined["IndoorRH_percent"] = combined["IndoorRH_frac"] * 100.0
if "Ambient2_F" not in combined.columns:
    combined["Ambient2_F"] = pd.NA

# 4) Keep only the columns we want in the final modeling table.
#    This enforces a stable schema before renaming.
keep_cols = [
    "comfort_timestamp_2022",
    "weather_timestamp",
    "source_file",
    "abs_gap_minutes",
    "Current Weather",
    "Temp (C)",
    "Humidity",
    "IndoorTemp_F",
    "IndoorRH_percent",
    "IndoorRH_frac",
    "WallTemp_F",
    "Ambient2_F",
    "R_wall",
    "A_envelope",
    "ACH",
    "volume",
    "RH_pred_frac",
    "WallTemp_pred_F",
    "BuildingHeatLoss_W",
    "UserHeatLoss_Wm2",
    "PhysicsVerdict",
    "Label",
]

missing_cols = [c for c in keep_cols if c not in combined.columns]
if missing_cols:
    raise ValueError(f"Missing expected columns before schema build: {missing_cols}")

combined = combined[keep_cols].copy()

# 5) Rename columns with source-provenance prefixes.
#    Prefix meaning:
#      meta_ = identifiers/timestamps/alignment metadata
#      wx_   = weather data
#      msr_  = collected/measured indoor data
#      bld_  = building constants
#      phy_  = steady-state model outputs
#      tgt_  = ML target label
rename_map = {
    "comfort_timestamp_2022": "meta_comfort_timestamp",
    "weather_timestamp": "meta_weather_timestamp",
    "source_file": "meta_source_file",
    "abs_gap_minutes": "meta_weather_gap_minutes",
    "Current Weather": "wx_condition",
    "Temp (C)": "wx_temp_c",
    "Humidity": "wx_rh_percent",
    "IndoorTemp_F": "msr_indoor_temp_f",
    "IndoorRH_percent": "msr_indoor_rh_percent",
    "IndoorRH_frac": "msr_indoor_rh_frac",
    "WallTemp_F": "msr_wall_temp_f",
    "Ambient2_F": "msr_ambient2_temp_f",
    "R_wall": "bld_r_wall",
    "A_envelope": "bld_a_envelope",
    "ACH": "bld_ach",
    "volume": "bld_volume",
    "RH_pred_frac": "phy_indoor_rh_frac",
    "WallTemp_pred_F": "phy_wall_temp_f",
    "BuildingHeatLoss_W": "phy_building_heat_loss_w",
    "UserHeatLoss_Wm2": "phy_user_heat_loss_wm2",
    "PhysicsVerdict": "phy_verdict",
    "Label": "tgt_comfort_label",
}
combined = combined.rename(columns=rename_map)

# 6) Standardize target label text and type.
combined["tgt_comfort_label"] = (
    combined["tgt_comfort_label"]
    .astype(str)
    .str.strip()
    .str.title()
    .replace({"Comfort": "Good"})
)

# 7) Enforce numeric dtypes on numeric features used by modeling.
numeric_cols = [
    "meta_weather_gap_minutes",
    "wx_temp_c",
    "wx_rh_percent",
    "msr_indoor_temp_f",
    "msr_indoor_rh_percent",
    "msr_indoor_rh_frac",
    "msr_wall_temp_f",
    "msr_ambient2_temp_f",
    "bld_r_wall",
    "bld_a_envelope",
    "bld_ach",
    "bld_volume",
    "phy_indoor_rh_frac",
    "phy_wall_temp_f",
    "phy_building_heat_loss_w",
    "phy_user_heat_loss_wm2",
]
for c in numeric_cols:
    combined[c] = pd.to_numeric(combined[c], errors="coerce")

# 8) Drop rows missing required training fields.
#    Keep weather condition and ambient2 optional.
required_cols = [
    "meta_comfort_timestamp",
    "tgt_comfort_label",
    "wx_temp_c",
    "wx_rh_percent",
    "msr_indoor_temp_f",
    "msr_indoor_rh_frac",
    "msr_wall_temp_f",
    "phy_indoor_rh_frac",
    "phy_wall_temp_f",
]
combined = combined.dropna(subset=required_cols).copy()

# 9) Remove exact duplicates for reproducibility.
combined = combined.drop_duplicates().reset_index(drop=True)

# 10) Save the final combined dataset for RF training.
combined_out = base / "rf_combined_dataset.csv"
combined.to_csv(combined_out, index=False)

# 11) Print QA summary so we can verify integrity before model training.
# print(f"Saved combined dataset: {combined_out.name}")
print(f"Rows: {len(combined):,}")
print(f"Columns: {combined.shape[1]}")

print("\nLabel distribution (target):")
print(combined["tgt_comfort_label"].value_counts(dropna=False))

print("\nMissing values by column (top 15):")
print(combined.isna().sum().sort_values(ascending=False).head(15))

# Check one-row-per-comfort-timestamp assumption.
dup_key_count = combined.duplicated(subset=["meta_comfort_timestamp", "meta_source_file"]).sum()
print(f"\nDuplicate rows on (meta_comfort_timestamp, meta_source_file): {dup_key_count}")

# Preview a few rows to confirm prefixes and values look correct.
display(combined.head(10))

Rows: 6,201
Columns: 22

Label distribution (target):
tgt_comfort_label
Good    4599
Cold    1106
Hot      496
Name: count, dtype: int64

Missing values by column (top 15):
meta_comfort_timestamp      0
meta_weather_timestamp      0
meta_source_file            0
meta_weather_gap_minutes    0
wx_condition                0
wx_temp_c                   0
wx_rh_percent               0
msr_indoor_temp_f           0
msr_indoor_rh_percent       0
msr_indoor_rh_frac          0
msr_wall_temp_f             0
msr_ambient2_temp_f         0
bld_r_wall                  0
bld_a_envelope              0
bld_ach                     0
dtype: int64

Duplicate rows on (meta_comfort_timestamp, meta_source_file): 0


,meta_comfort_timestamp,meta_weather_timestamp,meta_source_file,meta_weather_gap_minutes,wx_condition,wx_temp_c,wx_rh_percent,msr_indoor_temp_f,msr_indoor_rh_percent,msr_indoor_rh_frac,...,bld_r_wall,bld_a_envelope,bld_ach,bld_volume,phy_indoor_rh_frac,phy_wall_temp_f,phy_building_heat_loss_w,phy_user_heat_loss_wm2,phy_verdict,tgt_comfort_label
0,2022-03-28 23:50:08,2022-03-28 23:55:00,March28Data_W_IRRA(in).csv,4.866667,Clear,0.5,36.792179,68.54,26.9,0.269,...,2.3,70.0,0.5,30.0,0.097851,66.192968,702.103696,68.423423,Good,Good
1,2022-03-28 23:50:10,2022-03-28 23:55:00,March28Data_W_IRRA(in).csv,4.833333,Clear,0.5,36.792179,69.26,26.2,0.262,...,2.3,70.0,0.5,30.0,0.095465,66.865553,716.287609,66.697894,Good,Good
2,2022-03-28 23:50:12,2022-03-28 23:55:00,March28Data_W_IRRA(in).csv,4.800000,Clear,0.5,36.792179,69.26,26.2,0.262,...,2.3,70.0,0.5,30.0,0.095465,66.865553,716.287609,66.697894,Good,Good
3,2022-03-28 23:50:14,2022-03-28 23:55:00,March28Data_W_IRRA(in).csv,4.766667,Clear,0.5,36.792179,69.26,26.2,0.262,...,2.3,70.0,0.5,30.0,0.095465,66.865553,716.287609,66.697894,Good,Good
4,2022-03-28 23:50:16,2022-03-28 23:55:00,March28Data_W_IRRA(in).csv,4.733333,Clear,0.5,36.792179,69.26,26.2,0.262,...,2.3,70.0,0.5,30.0,0.095465,66.865553,716.287609,66.697894,Good,Good
5,2022-03-28 23:50:18,2022-03-28 23:55:00,March28Data_W_IRRA(in).csv,4.700000,Clear,0.5,36.792179,69.26,26.2,0.262,...,2.3,70.0,0.5,30.0,0.095465,66.865553,716.287609,66.697894,Good,Good
6,2022-03-28 23:50:20,2022-03-28 23:55:00,March28Data_W_IRRA(in).csv,4.666667,Clear,0.5,36.792179,69.26,26.2,0.262,...,2.3,70.0,0.5,30.0,0.095465,66.865553,716.287609,66.697894,Good,Good
7,2022-03-28 23:50:22,2022-03-28 23:55:00,March28Data_W_IRRA(in).csv,4.633333,Clear,0.5,36.792179,69.26,26.2,0.262,...,2.3,70.0,0.5,30.0,0.095465,66.865553,716.287609,66.697894,Good,Good
8,2022-03-28 23:50:24,2022-03-28 23:55:00,March28Data_W_IRRA(in).csv,4.600000,Clear,0.5,36.792179,69.26,26.1,0.261,...,2.3,70.0,0.5,30.0,0.095465,66.865553,716.287609,66.697894,Good,Good
9,2022-03-28 23:50:26,2022-03-28 23:55:00,March28Data_W_IRRA(in).csv,4.566667,Clear,0.5,36.792179,69.26,26.1,0.261,...,2.3,70.0,0.5,30.0,0.095465,66.865553,716.287609,66.697894,Good,Good
